# Tutorial 04: Advanced Metadata Handling

Do you have a massive Excel sheet with drug treatments? Does your microscope save different colors as completely separate files? Are your folders named in overly complex ways? This notebook fixes that.

This tutorial covers how to handle complex dataset structures:
1. **CSV Metadata Lookup**: Labeling your images using an external CSV/Excel sheet.
2. **Multi-channel Imaging**: Stitching together samples where each fluorescent color was saved as a separate file.
3. **Filtering & Extraction**: Creating small rules to ignore bad images or extract text from messy file names.

---

## 1. Setup

Import core classes and utilities. We also use `pandas` for handling external spreadsheets.

In [ ]:
import pandas as pd
from phenome import PhenoMe, load_dinov2_model, make_dataframe_metadata_fn

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pheno = PhenoMe(device=device, seed=42)

## 2. Using an external CSV for Metadata

Instead of encoding metadata in folder names, you might have a master spreadsheet. This is common when labels are generated after image capture (e.g., from an automated assay or manual curation).

**How this interacts with `find_files`:** The pipeline first scans your image folder, then looks up each discovered file in the CSV by filename. **Only the overlap is used**—rows in the CSV with no matching file on disk are ignored, and image files without a matching row are dropped by default (`on_missing_metadata='drop'`). So your `file_df` length may differ from the number of CSV rows.


In [ ]:
# Dummy metadata dataframe
df = pd.DataFrame({
    'filename': ['image_001', 'image_002', 'image_003'],
    'treatment': ['Control', 'DrugA', 'DrugB'],
    'dose': [0, 10, 100],
    'replicate': [1, 1, 1]
})

# Create a lookup function
# filename_column is the column in your CSV that matches the actual file names
metadata_fn = make_dataframe_metadata_fn(df, filename_column='filename')

# When finding files, use metadata_fn to attach 'treatment' and 'dose' to each image
file_df = pheno.find_files("path/to/images", metadata_fn=metadata_fn)

print(file_df[['img_path', 'treatment', 'dose']].head())

## 3. Multi-channel (Split Files) Workflow

Some microscopes save different biological features down to completely separate files—for example, the nucleus (blue) in one image and the cell body (red) in another. **PhenoMe** can easily stitch these separate files back together into a single sample for analysis.

In [ ]:
# CSV where each row is a physical sample, but channels are in separate files
df_channels = pd.DataFrame({
    'dna': ['wellA1_ch0.tif', 'wellA2_ch0.tif'],
    'actin': ['wellA1_ch1.tif', 'wellA2_ch1.tif'],
    'tubulin': ['wellA1_ch2.tif', 'wellA2_ch2.tif'],
    'condition': ['Control', 'Stimulated']
})

# Tell PhenoMe which columns hold the file names for each channel
# The order in the list specifies the channel order [0, 1, 2, ...]
channel_cols = ['dna', 'actin', 'tubulin']
metadata_fn = make_dataframe_metadata_fn(df_channels, filename_column=channel_cols)

# find_files will automatically group these paths into a single 'sample'
file_df = pheno.find_files("path/to/images", metadata_fn=metadata_fn)

# When processing, 'split' mode ensures multi-channel stacks are built correctly
# wrapper = load_dinov2_model(device=device)
# pheno.process_images(wrapper, channel_mode="split")

## 4. Filtering Out Bad Images and Complex Folder Names

Sometimes you have completely unreadable files, random thumbnails, or messy file names (like `Image_P01_W23.tif`). You can use simple rules to tell the pipeline exactly what to skip and how to read the information it actually needs.

In [ ]:
import os

# 1. Example: Only find .png files and skip anything with 'thumb' in the name
file_df = pheno.find_files(
    "path/to/data",
    extensions=[".png"]
    # Internal logic skips files that return None from metadata_fn
)

# 2. Example: Custom metadata extractor for complex filenames
def parse_plate_well(path):
    """(path) -> dict mapping names to values from weird filename."""
    fname = os.path.basename(path)
    if not fname.startswith("Image"): 
        return None # Ignore this file entirely
    
    # Extract pattern like Image_P01_W23.tif
    parts = fname.replace(".tif", "").split("_")
    return {
        "plate": parts[1],
        "well": parts[2],
        "batch": "v2"
    }

file_df = pheno.find_files("data/", metadata_fn=parse_plate_well)
print(f"Found {len(file_df)} images with custom extraction")


## Next Steps

- [03_core_phenotyping_workflow.ipynb](03_core_phenotyping_workflow.ipynb): Core API tutorial
- [05_exploratory_analysis_and_explainability.ipynb](05_exploratory_analysis_and_explainability.ipynb): Advanced clustering and explainability
- [06_extending_phenome_plugins.ipynb](06_extending_phenome_plugins.ipynb): Writing custom plugins and model wrappers